# Quick Start: Data Processing Pipeline

This notebook demonstrates how to use the new data processing pipeline to:
1. Process all 5 datasets with one function call
2. Merge them into a master dataset
3. Export for analysis

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Repository root: {REPO_ROOT}")

## Step 1: Process All Datasets

Process all 5 datasets (CPDS, Population, GDP, Inflation, Dependency) with validation.

In [ ]:
from clean import process_all_datasets

# Process everything with one call!
results = process_all_datasets(
    repo_root=REPO_ROOT,
    year_min=1980,
    year_max=2023,
    validate=True,
    save_outputs=True
)

## Step 2: Inspect Individual Datasets

In [ ]:
# Quick overview
for name, df in results.items():
    if df is not None:
        print(f"\n{name.upper()}:")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  Countries: {df['iso3'].nunique()}")
        print(f"  Year range: {df['year'].min()}-{df['year'].max()}")

In [ ]:
# View sample data
results['cpds'].head()

## Step 3: Merge All Datasets

Merge all datasets on (iso3, year) to create one master dataset for analysis.

In [ ]:
from clean import merge_all_datasets

# Merge everything
master = merge_all_datasets(
    results,
    how='outer'  # Keep all observations from all datasets
)

In [ ]:
# View the merged dataset
print(f"Master dataset shape: {master.shape}")
print(f"\nColumns: {list(master.columns)}")
master.head(10)

## Step 4: Check Data Coverage

In [ ]:
from clean import get_merge_summary

# Get coverage summary by country
coverage = get_merge_summary(master)
coverage.head(10)

In [ ]:
# Check missing data
print("Missing values by variable:")
print(master.isnull().sum())

## Step 5: Save Master Dataset

In [ ]:
from clean import save_master_dataset

# Save in multiple formats
saved = save_master_dataset(
    master,
    output_path=REPO_ROOT / "data" / "final" / "master_dataset",
    formats=['parquet', 'csv']
)

print("\nSaved files:")
for fmt, path in saved.items():
    print(f"  {fmt}: {path}")

## Quick Analysis Example

In [ ]:
# Example: Summary statistics for key variables
key_vars = ['sstran', 'deficit', 'debt', 'ln_population', 'ln_gdppc', 'inflation_cpi', 'dependency_ratio']
master[key_vars].describe()

In [ ]:
# Example: Check data for specific country
usa_data = master[master['iso3'] == 'USA'].sort_values('year')
usa_data.head(10)

## Next Steps

Now you have a clean master dataset ready for analysis!

You can:
- Run regressions
- Create visualizations
- Perform time series analysis
- Export to Stata for further analysis

The master dataset includes:
- `iso3`, `year`: Identifiers
- `sstran`, `deficit`, `debt`: From CPDS
- `ln_population`: Natural log of population
- `ln_gdppc`: Natural log of GDP per capita
- `inflation_cpi`: Inflation rate
- `dependency_ratio`: Age dependency ratio